# 01 — Accuracy comparisons with McNemar-style power analysis

Use this notebook when the evaluation output is **paired binary correctness**:

- Model A correct or incorrect on each test item.
- Model B correct or incorrect on the same test items.

This is common for classification, QA exact match, and other accuracy-like metrics.

The key point: the expected final accuracies are not enough. You also need an assumption about **agreement**: how often the two models have the same correctness outcome.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from power_utils import *


## Student input

Run the cell below and enter your assumptions. Press Enter to accept the suggested default.

Definitions:

- `n_instances`: number of evaluation examples.
- `baseline_accuracy`: expected accuracy of Model A.
- `expected_delta`: expected improvement of Model B over Model A. A 1 percentage-point improvement is `0.01`.
- `agreement_rate`: expected proportion of examples where A and B are either both correct or both wrong.
- `alpha`: significance threshold.
- `target_power`: desired probability of detecting the effect if it is real.


In [ ]:
n_instances = ask_int("Number of test instances", 1000, minimum=20)
baseline_accuracy = ask_float("Expected accuracy of Model A", 0.90, minimum=0.0, maximum=1.0)
expected_delta = ask_float("Expected accuracy improvement of Model B over A", 0.01, minimum=0.0, maximum=1.0)
agreement_rate = ask_float("Expected agreement rate in correctness outcomes", 0.95, minimum=0.0, maximum=1.0)
alpha = ask_float("Significance threshold alpha", 0.05, minimum=0.0001, maximum=0.5)
target_power = ask_float("Target power", 0.80, minimum=0.01, maximum=0.99)
n_trials = ask_int("Number of simulation trials", 2000, minimum=100)

params = dict(
    n_instances=n_instances,
    baseline_accuracy=baseline_accuracy,
    expected_delta=expected_delta,
    agreement_rate=agreement_rate,
    alpha=alpha,
    target_power=target_power,
    n_trials=n_trials,
)
params

## Check what these assumptions imply

The simulation internally uses four possible outcomes for each item:

| Outcome | Meaning |
|---|---|
| both correct | A and B are both correct |
| only A correct | A is correct, B is wrong |
| only B correct | B is correct, A is wrong |
| both wrong | A and B are both wrong |

McNemar's test uses the two disagreement cells: **only A correct** and **only B correct**.


In [ ]:
probs = accuracy_category_probabilities(
    baseline_accuracy=baseline_accuracy,
    expected_delta=expected_delta,
    agreement_rate=agreement_rate,
)

pd.DataFrame({
    "outcome": ["both correct", "only A correct", "only B correct", "both wrong"],
    "probability": probs,
    "expected_count": probs * n_instances,
})

## Estimate power for this test size

In [ ]:
power = estimate_mcnemar_power(
    n_instances=n_instances,
    baseline_accuracy=baseline_accuracy,
    expected_delta=expected_delta,
    agreement_rate=agreement_rate,
    alpha=alpha,
    n_trials=n_trials,
)

print(f"Estimated power: {power:.3f}")
print(f"Interpretation: if the true improvement is {expected_delta:.1%}, this design detects it about {power:.1%} of the time.")

## Estimate required test-set size

In [ ]:
required_n, achieved_power = find_required_n_mcnemar(
    baseline_accuracy=baseline_accuracy,
    expected_delta=expected_delta,
    agreement_rate=agreement_rate,
    alpha=alpha,
    target_power=target_power,
    n_trials=n_trials,
)

if required_n is None:
    print("No required n found within the search range. Try a larger effect size or lower target power.")
else:
    print(f"Required test instances for {target_power:.0%} power: {required_n}")
    print(f"Estimated power at that n: {achieved_power:.3f}")

## Compare power across dataset sizes

This cell uses the same assumptions and changes only the number of test instances.


In [ ]:
n_grid = np.array([100, 250, 500, 1000, 2000, 5000, 10000])
rows = []
for n in n_grid:
    rows.append({
        "n_instances": n,
        "power": estimate_mcnemar_power(
            n_instances=int(n),
            baseline_accuracy=baseline_accuracy,
            expected_delta=expected_delta,
            agreement_rate=agreement_rate,
            alpha=alpha,
            n_trials=max(500, n_trials // 2),
        ),
    })
power_by_n = pd.DataFrame(rows)
power_by_n

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(power_by_n["n_instances"], power_by_n["power"], marker="o")
plt.axhline(target_power, linestyle="--")
plt.xscale("log")
plt.xlabel("Number of test instances")
plt.ylabel("Estimated power")
plt.title("Power increases with test-set size")
plt.show()

## Optional: relate this to GLUE/SQuAD-style test sizes

This does not reproduce the paper's full analysis. It simply lets students apply their own assumptions to common test-set sizes.


In [ ]:
glue_sizes = pd.read_csv(DATA_DIR / "glue" / "glue_task_test_set_sizes.csv")
glue_sizes

In [ ]:
rows = []
for _, row in glue_sizes.iterrows():
    n = int(row["size"])
    pwr = estimate_mcnemar_power(
        n_instances=n,
        baseline_accuracy=baseline_accuracy,
        expected_delta=expected_delta,
        agreement_rate=agreement_rate,
        alpha=alpha,
        n_trials=max(500, n_trials // 2),
    )
    rows.append({"dataset": row["Dataset"], "n_instances": n, "power_under_your_assumptions": pwr})

pd.DataFrame(rows).sort_values("n_instances")